
# Data Preprocessing

Real-world data is messy. It contains missing values, text labels, and features on vastly different scales. **Data Preprocessing** is the process of converting raw data into a clean dataset that machine learning algorithms can understand.

Without this step, even the most powerful algorithms will fail. The most common tasks include:

1.  **Handling Missing Values**: Data holes cause errors. We can drop them or fill them (Imputation).
2.  **Feature Scaling**: Algorithms like KNN and SVM calculate "distance." If one feature is in thousands (Salary) and another is small (Age), the large one will dominate. Scaling fixes this.
3.  **Encoding Categorical Variables**: Computers speak math, not English. We must convert text labels (e.g., "Male", "Female") into numbers.

## Practical Demonstration: Titanic Survival

We will use the famous Titanic dataset to demonstrate these steps. We want to predict if a passenger survived based on their Fare and Sex.

### Load and Inspect

In [ ]:
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np

# Load data
data = fetch_openml('titanic', version=1, as_frame=True, parser="auto")
df = data.frame

# Keep only relevant columns for this demo
df = df[['fare', 'sex', 'age', 'survived']]

print(df.info())
print("\nFirst 5 rows:")
print(df.head())

### Exploratory Data Analysis (EDA)

Before changing the data, we visualize it.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set style
sns.set_theme(style="whitegrid")

plt.figure(figsize=(12, 5))

# Plot 1: Survival by Fare
plt.subplot(1, 3, 1)
sns.boxplot(x='fare', y='survived', data=df, orient='horizontal')
plt.xscale('log')
plt.title('Survival Distribution by Fare')

# Plot 2: Survival by Age
plt.subplot(1, 3, 2)
sns.boxplot(x='age', y='survived', data=df, orient='horizontal')
plt.title('Survival Distribution by Age')

# Plot 3: Survival by Gender
plt.subplot(1, 3, 3)
sns.countplot(x='sex', hue='survived', data=df)
plt.title('Survival Count by Gender')

plt.tight_layout()
plt.show()

### Train-Test Split

**Crucial Rule**: Apply preprocessing **after** splitting the data. If you calculate the mean for imputation using the whole dataset, you are leaking information from the Test set into the Training set ("Data Leakage").

In [ ]:
from sklearn.model_selection import train_test_split

X = df[['fare', 'sex', 'age']]
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Preprocessing Steps

#### Handling Missing Values (Imputation)

All columns have some missing values, which we have to deal with. We will do so by *imputing* the missing values in the following way: for *numerical* columns (`fare` and `age`), we fill in the missing values with the average of the respective column in the **training set**. For *categorical* variables (`sex`), we will fill in the missing values with the most frequent value from the **training set**.

In [ ]:
from sklearn.impute import SimpleImputer

# Create numerical imputer, train on TRAIN, transform TRAIN and TEST
num_imputer = SimpleImputer(strategy='mean')
X_train[['fare', 'age']] = num_imputer.fit_transform(X_train[['fare', 'age']])
X_test[['fare', 'age']] = num_imputer.transform(X_test[['fare', 'age']])

# Create categorical imputer
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[['sex']] = cat_imputer.fit_transform(X_train[['sex']])
X_test[['sex']] = cat_imputer.transform(X_test[['sex']])

print("Missing values in Train after imputation:", X_train[['fare', 'age', 'sex']].isnull().sum())

#### Feature Scaling (Standardization)

We use `StandardScaler` to force the `fare` and `age` to have a mean of 0 and a variance of 1, which this helps the model converge faster.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit on TRAIN, transform on TRAIN and TEST
X_train[['fare', 'age']] = scaler.fit_transform(X_train[['fare', 'age']])
X_test[['fare', 'age']] = scaler.transform(X_test[['fare', 'age']])

print(X_train[['fare', 'age']].head())

#### Encoding Categoricals (One-Hot Encoding)

The `sex` column is text. We use One-Hot Encoding to create a binary column (e.g., `sex_male`).

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# drop='first' removes one column to avoid collinearity (dummy variable trap)
# If 'female' is 0, 'male' must be 1. We don't need two columns.
encoder = OneHotEncoder(drop='first', sparse_output=False)

# Fit and transform
sex_train_encoded = encoder.fit_transform(X_train[['sex']])
sex_test_encoded = encoder.transform(X_test[['sex']])

# Convert back to DataFrame to keep things readable
feature_names = encoder.get_feature_names_out(['sex'])

df_train_sex = pd.DataFrame(sex_train_encoded, columns=feature_names, index=X_train.index)
df_test_sex = pd.DataFrame(sex_test_encoded, columns=feature_names, index=X_test.index)

# Concatenate back to main dataframe and drop original 'sex' column
X_train_final = pd.concat([X_train.drop(columns=['sex']), df_train_sex], axis=1)
X_test_final = pd.concat([X_test.drop(columns=['sex']), df_test_sex], axis=1)

print(X_train_final.head())

### Training and Evaluation

We train a **Logistic Regression** model (used for classification).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Train
model = LogisticRegression()
model.fit(X_train_final, y_train)

# Predict
y_pred = model.predict(X_test_final)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Died', 'Survived'], yticklabels=['Died', 'Survived'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## Exercises

Apply these techniques to the **Iris Dataset**. **Twist**: Instead of classification, perform a **Regression** task. Predict the `petal length` using the other features.

### Setup

-   Load the Iris dataset using `load_iris`.
-   Convert it to a DataFrame.
-   Define `X` (Sepal Length, Sepal Width, Petal Width, Species) and `y` (Petal Length).

### Split and Preprocess

1.  Split the data (80/20).
2.  Scale the numerical columns (`sepal length`, `sepal width`, `petal width`).
3.  One-Hot Encode the `species_name` column.

### Train Linear Regression

Train a Linear Regression model and verify the Mean Squared Error (MSE).

## Summary

Preprocessing is the "plumbing" of Machine Learning. It's not glamorous, but it is essential.

-   We handled **Missing Values** with Imputation.
-   We normalized ranges with **Scaling**.
-   We translated text to numbers with **One-Hot Encoding**.